# 準備演習 04: マルチエージェントリサーチパイプラインの設計とデバッグ

## 目的

- coordinator / subagent の役割分離を理解する
- Task 系ツールを前提とした並列委譲、明示的なコンテキスト受け渡し、structured error を確認する
- provenance を保持した統合と、矛盾する証拠を残したままの synthesis を体験する

## 対象ドメイン

- Domain 1: Agentic Architecture & Orchestration
- Domain 5: Context Management & Reliability

## 完成イメージ

この Notebook は Claude Agent SDK ベースの multi-agent research を **小さな coordinator** として観察する教材です。
最新の subagent orchestration は公式ドキュメントと完成版 Lab [../labs/04-multi-agent-research/](../labs/04-multi-agent-research/) を参照してください。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import asyncio
import time
from pprint import pprint


def section(title: str):
    print(f"\n=== {title} ===")

@dataclass
class SubagentResult:
    claim: str
    evidence_excerpt: str
    source: str
    published_at: str
    confidence: float

@dataclass
class SubagentError:
    agent: str
    error_type: str
    attempted_query: str
    partial_result: str | None
    is_retryable: bool

## Step 1. coordinator と subagent の prompt を分ける

サブエージェントは自動コンテキスト継承に頼らず、**必要な発見を prompt に明示的に渡す** 前提にします。

In [ ]:
TASK_TOOL_PLAN = {
    "coordinator": "Task-style delegation to specialized subagents",
    "subagents": ["web-researcher", "docs-analyst"],
}


def build_subagent_prompt(query: str, prior_findings: list[str], source_type: str) -> str:
    joined = "\n".join(f"- {item}" for item in prior_findings) or "- まだ発見なし"
    return (
        f"query: {query}\n"
        f"source_type: {source_type}\n"
        "execution_model: launch as a focused Task-style subagent\n"
        "prior_findings:\n"
        f"{joined}\n\n"
        "主張・証拠抜粋・source・published_at を含む構造化出力だけを返してください。"
    )

section("Task tool 前提の設計")
pprint(TASK_TOOL_PLAN)
print(build_subagent_prompt("plan mode の使い方", ["既に docs を確認済み"], "web_search"))

## Step 2. 並列サブエージェント実行を観察する

In [ ]:
async def mock_subagent(name: str, delay: float, claim: str) -> SubagentResult:
    await asyncio.sleep(delay)
    return SubagentResult(
        claim=claim,
        evidence_excerpt=f"{name} evidence excerpt",
        source=f"mock://{name}",
        published_at="2026-03-26",
        confidence=0.8,
    )

async def compare_latency():
    start = time.perf_counter()
    sequential = [
        await mock_subagent("web", 0.3, "web says plan mode is read-only"),
        await mock_subagent("docs", 0.2, "docs say plan mode is safer for large changes"),
    ]
    sequential_time = time.perf_counter() - start

    start = time.perf_counter()
    parallel = await asyncio.gather(
        mock_subagent("web", 0.3, "web says plan mode is read-only"),
        mock_subagent("docs", 0.2, "docs say plan mode is safer for large changes"),
    )
    parallel_time = time.perf_counter() - start
    return sequential, sequential_time, parallel, parallel_time

sequential, sequential_time, parallel, parallel_time = asyncio.run(compare_latency())
section("latency comparison")
print(f"sequential: {sequential_time:.3f}s")
print(f"parallel:   {parallel_time:.3f}s")

### 確認ポイント

- 1 回のレスポンス内で複数 Task を投げられると、独立した subagent は並列化しやすい
- 逐次実行とのレイテンシ差を観察できる

## Step 3. structured output と provenance を保つ

In [ ]:
section("structured subagent results")
for result in parallel:
    pprint(result)

## Step 4. タイムアウトなどの失敗を coordinator へ伝播する

In [ ]:
async def flaky_subagent(name: str):
    if name == "web":
        raise TimeoutError("web subagent timed out")
    return SubagentResult(
        claim="docs result survived",
        evidence_excerpt="official docs excerpt",
        source="mock://docs",
        published_at="2026-03-26",
        confidence=0.84,
    )

async def collect_with_partial_failure():
    tasks = [flaky_subagent("web"), flaky_subagent("docs")]
    raw = await asyncio.gather(*tasks, return_exceptions=True)
    successes, failures = [], []
    for agent_name, item in zip(["web", "docs"], raw):
        if isinstance(item, Exception):
            failures.append(SubagentError(agent_name, "timeout", "plan mode の使い方", None, True))
        else:
            successes.append(item)
    return successes, failures

successes, failures = asyncio.run(collect_with_partial_failure())
section("partial failure handling")
pprint(successes)
pprint(failures)

## Step 5. 矛盾する証拠を片方消さずに統合する

In [ ]:
conflicting = [
    SubagentResult(
        claim="Source A: default permission mode can be set to plan",
        evidence_excerpt="plansDirectory and defaultMode are configurable",
        source="mock://source-a",
        published_at="2026-03-25",
        confidence=0.78,
    ),
    SubagentResult(
        claim="Source B: session-level switching via Shift+Tab is the primary documented path",
        evidence_excerpt="common workflows emphasizes interactive switching",
        source="mock://source-b",
        published_at="2026-03-26",
        confidence=0.83,
    ),
]

report = {
    "established_findings": [conflicting[1].claim],
    "contested_findings": [
        {"claim": item.claim, "source": item.source, "published_at": item.published_at}
        for item in conflicting
    ],
}

section("contradictory evidence report")
pprint(report)

## 完成版 Lab 参照

- Notebook では coordinator / subagent 設計 → 並列化 → structured output → error propagation → contradictory evidence handling の順に学習しました
- 完成版 Lab は Claude Agent SDK の `query()`、`AgentDefinition`、より現実的な synthesis フローを含む reference implementation です
- 詳細は [../labs/04-multi-agent-research/](../labs/04-multi-agent-research/) を参照してください
- subagent orchestration の現在の推奨パターンは、必ず最新の公式ドキュメントで確認してください